# 🔄 Secondary Structure Analysis

---

## Learning Objectives

- Understand α-helix and β-sheet structures
- Use DSSP for secondary structure assignment
- Visualize secondary structure in Jmol/PyMOL
- Parse HELIX and SHEET records from PDB files

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Secondary Structure Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    PROTEIN SECONDARY STRUCTURE                          │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  α-HELIX                           β-SHEET                              │
│                                                                         │
│    ╭─────╮                         ───────►  strand 1                   │
│   ╱  ╭───╲                         ◄───────  strand 2  (antiparallel)   │
│  │  ╱ ╭───│                        ───────►  strand 3                   │
│  │ │ ╱ ╭──│                                                             │
│  │ │ │╱ ╭─│         OR:           ───────►  strand 1                   │
│  │ │ │ │╱ │                        ───────►  strand 2  (parallel)       │
│  ╰─────────╯                       ───────►  strand 3                   │
│                                                                         │
│  3.6 residues/turn                 H-bonds between strands              │
│  H-bonds: i → i+4                  Pleated appearance                   │
│  Right-handed                                                           │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## DSSP Secondary Structure Codes

| Code | Structure | Description |
|------|-----------|-------------|
| H | α-helix | 4-turn helix (i→i+4 H-bonds) |
| B | β-bridge | Single residue β-bridge |
| E | β-strand | Extended strand in β-sheet |
| G | 3₁₀-helix | 3-turn helix (i→i+3) |
| I | π-helix | 5-turn helix (i→i+5) |
| T | Turn | H-bonded turn |
| S | Bend | High curvature |
| - | Coil | No regular structure |

```
    Helix Types:
    
    3₁₀ helix (G)    α-helix (H)      π-helix (I)
    i → i+3          i → i+4          i → i+5
    tighter          most common      looser
    
    ┌───┐            ┌────┐           ┌─────┐
    │ 3 │            │ 3.6│           │ 4.4 │  residues/turn
    └───┘            └────┘           └─────┘
```

In [ ]:
def parse_secondary_structure(pdb_file):
    """
    Parse HELIX and SHEET records from PDB file.
    
    Returns:
        dict with 'helices' and 'sheets' lists
    """
    helices = []
    sheets = []
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('HELIX'):
                helix = {
                    'serial': int(line[7:10]),
                    'helix_id': line[11:14].strip(),
                    'init_res': line[15:18].strip(),
                    'init_chain': line[19],
                    'init_seq': int(line[21:25]),
                    'end_res': line[27:30].strip(),
                    'end_chain': line[31],
                    'end_seq': int(line[33:37]),
                    'helix_class': int(line[38:40]),
                    'length': int(line[71:76]) if len(line) > 71 else None
                }
                helices.append(helix)
                
            elif line.startswith('SHEET'):
                sheet = {
                    'strand': int(line[7:10]),
                    'sheet_id': line[11:14].strip(),
                    'num_strands': int(line[14:16]),
                    'init_res': line[17:20].strip(),
                    'init_chain': line[21],
                    'init_seq': int(line[22:26]),
                    'end_res': line[28:31].strip(),
                    'end_chain': line[32],
                    'end_seq': int(line[33:37]),
                    'sense': int(line[38:40]) if line[38:40].strip() else 0
                }
                sheets.append(sheet)
    
    return {'helices': helices, 'sheets': sheets}

# Helix class codes
HELIX_CLASSES = {
    1: "Right-handed α-helix",
    2: "Right-handed ω-helix",
    3: "Right-handed π-helix",
    4: "Right-handed γ-helix",
    5: "Right-handed 3₁₀-helix",
    6: "Left-handed α-helix",
    7: "Left-handed ω-helix",
    8: "Left-handed γ-helix",
    9: "2₇ ribbon/helix",
    10: "Polyproline"
}

print("Secondary structure parser defined.")
print("\nHelix class codes:")
for code, name in HELIX_CLASSES.items():
    print(f"  {code}: {name}")

---

## Ramachandran Plot

The Ramachandran plot shows allowed φ/ψ backbone angles.

```
                          ψ (psi)
                           180°
                            │
                  β-sheet   │   
                   region   │   
         ┌─────────────────┐│
         │    ████████     ││
         │   ██████████    ││
   -180° ├───────────────────┼─── 180° φ (phi)
         │                  ││
         │     ████████    ││ α-helix
         │    ██████████   ││  region
         └─────────────────┘│
                            │
                          -180°
                          
    ████ = Allowed regions (>99% of residues)
    
    Typical values:
    α-helix:  φ = -57°, ψ = -47°
    β-sheet:  φ = -120°, ψ = +120°
    Left α:   φ = +60°, ψ = +60° (glycine only)
```

In [ ]:
import math

def calculate_dihedral(p1, p2, p3, p4):
    """
    Calculate dihedral angle between four points.
    
    Parameters:
        p1, p2, p3, p4: (x, y, z) tuples for four atoms
        
    Returns:
        Dihedral angle in degrees
    """
    import numpy as np
    
    p1, p2, p3, p4 = [np.array(p) for p in [p1, p2, p3, p4]]
    
    b1 = p2 - p1
    b2 = p3 - p2
    b3 = p4 - p3
    
    # Normal vectors
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    
    # Normalize
    n1 = n1 / np.linalg.norm(n1)
    n2 = n2 / np.linalg.norm(n2)
    
    # Calculate angle
    m1 = np.cross(n1, b2 / np.linalg.norm(b2))
    x = np.dot(n1, n2)
    y = np.dot(m1, n2)
    
    return math.degrees(math.atan2(y, x))

def calculate_phi_psi(backbone_atoms):
    """
    Calculate φ (phi) and ψ (psi) angles for each residue.
    
    φ: C(i-1) - N(i) - CA(i) - C(i)
    ψ: N(i) - CA(i) - C(i) - N(i+1)
    """
    angles = []
    
    # Group atoms by residue
    residues = {}
    for atom in backbone_atoms:
        key = (atom['chain_id'], atom['res_seq'])
        if key not in residues:
            residues[key] = {}
        residues[key][atom['atom_name']] = (atom['x'], atom['y'], atom['z'])
    
    # Sort by residue number
    sorted_keys = sorted(residues.keys(), key=lambda x: x[1])
    
    for i in range(1, len(sorted_keys) - 1):
        prev_key = sorted_keys[i-1]
        curr_key = sorted_keys[i]
        next_key = sorted_keys[i+1]
        
        prev_res = residues[prev_key]
        curr_res = residues[curr_key]
        next_res = residues[next_key]
        
        if all(k in prev_res for k in ['C']) and \
           all(k in curr_res for k in ['N', 'CA', 'C']) and \
           all(k in next_res for k in ['N']):
            
            phi = calculate_dihedral(
                prev_res['C'], curr_res['N'], curr_res['CA'], curr_res['C']
            )
            psi = calculate_dihedral(
                curr_res['N'], curr_res['CA'], curr_res['C'], next_res['N']
            )
            
            angles.append({
                'residue': curr_key,
                'phi': phi,
                'psi': psi
            })
    
    return angles

print("Phi/Psi calculation functions defined.")
print("\nTypical backbone angles:")
print("  α-helix: φ ≈ -57°, ψ ≈ -47°")
print("  β-sheet: φ ≈ -120°, ψ ≈ +120°")

---

## Jmol Secondary Structure Visualization

```javascript
// Load structure
load =1crn

// Cartoon representation (shows secondary structure)
cartoon only

// Color by secondary structure
color structure

// Or custom colors
select helix
color red
select sheet
color yellow
select coil
color grey

// Show ribbon instead
ribbons only

// Show backbone trace
trace only
color chain
```

---

## 🏋️ Exercises

### Exercise 1: Secondary Structure Content
Calculate the percentage of α-helix, β-sheet, and coil in a protein.

### Exercise 2: Ramachandran Plot
Generate a Ramachandran plot from backbone coordinates.

### Exercise 3: Compare Structures
Compare secondary structure assignments between different methods (PDB vs DSSP).

---

## 📚 Resources

- [DSSP Algorithm](https://swift.cmbi.umcn.nl/gv/dssp/)
- [MolProbity Validation](http://molprobity.biochem.duke.edu/)

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/04_Secondary_Structure/01_Secondary_Structure.ipynb`)
